In [1]:
# Imports
import random
import time

import librosa
import numpy as np
import torch
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from transformers import VoxtralRealtimeProcessor, VoxtralRealtimeForConditionalGeneration

# Dispositivo = GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cuda:0


In [2]:
# Cargar modelo y procesador
repo_id = "mistralai/Voxtral-Mini-4B-Realtime-2602"
print(f"Cargando modelo y procesador: {repo_id}")

processor = VoxtralRealtimeProcessor.from_pretrained(repo_id)
model = VoxtralRealtimeForConditionalGeneration.from_pretrained(repo_id).to(device)
model.eval()

audio_config = processor.mistral_common_audio_config
audio_config.transcription_delay_ms = 480

sr = processor.feature_extractor.sampling_rate
hop = processor.feature_extractor.hop_length
win = processor.feature_extractor.win_length

print(f"Retardo: {audio_config.transcription_delay_ms} ms")

Cargando modelo y procesador: mistralai/Voxtral-Mini-4B-Realtime-2602


Loading weights:   0%|          | 0/711 [00:00<?, ?it/s]

Retardo: 480 ms


In [3]:
N_AUDIOS = 20
SEMILLA = 42
DURACION_MAX_S = 90

IDIOMAS = {
    "es": ("../../Europarl-ST/es/en/test", "Europarl-ST.v2.es.en.test.lst", "Spanish"),
    "en": ("../../Europarl-ST/en/es/test", "Europarl-ST.v2.en.es.test.lst", "English"),
    "fr": ("../../Europarl-ST/fr/en/test", "Europarl-ST.fr.en.test.lst", "French"),
    "de": ("../../Europarl-ST/de/en/test", "Europarl-ST.de.en.test.lst", "German"),
}


def muestra(idioma):
    """Barajado con semilla fija; se toman los primeros que no exceden el tope.

    El tope existe porque el contexto de Qwen se agota en torno a los 120 s con
    bloques de 480 ms, y su coste crece de forma cuadratica con la duracion.
    """
    base, lst, _ = IDIOMAS[idioma]
    with open(base + "/" + lst, "r", encoding="utf-8") as f:
        todos = [l.strip() for l in f if l.strip()]

    random.Random(SEMILLA).shuffle(todos)

    sel = []
    for a in todos:
        d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
        if d <= DURACION_MAX_S:
            sel.append(a)
            if len(sel) == N_AUDIOS:
                break
    return sel


seleccion = {k: muestra(k) for k in IDIOMAS}
for k, v in seleccion.items():
    print(k, len(v), "audios")

/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duratio

es 20 audios
en 20 audios
fr 20 audios
de 20 audios


/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)
/tmp/ipykernel_5878/3631658208.py:27: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  d = librosa.get_duration(path=base + "/%s/audio_clip_diarization.m4a" % a)


In [4]:
def transcribir_offline(audio_array):
    inputs = processor(audio_array, return_tensors="pt").to(device)
    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(model.dtype)

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(**inputs)
    total = time.perf_counter() - t0

    return processor.batch_decode(outputs, skip_special_tokens=True)[0], total


def transcribir_streaming(audio_array):
    npad = processor.num_right_pad_tokens
    npad = npad() if callable(npad) else npad
    audio_array = np.pad(audio_array, (0, npad * processor.raw_audio_length_per_tok))

    entrada = processor(
        audio_array[:processor.num_samples_first_audio_chunk],
        is_streaming=True,
        is_first_audio_chunk=True,
        return_tensors="pt",
    ).to(model.device, dtype=model.dtype)

    trozos = [entrada.input_features]
    mel = processor.num_mel_frames_first_audio_chunk
    start = mel * hop - win // 2

    while (end := start + processor.num_samples_per_audio_chunk) < audio_array.shape[0]:
        t = processor(
            audio_array[start:end],
            is_streaming=True,
            is_first_audio_chunk=False,
            return_tensors="pt",
        )
        trozos.append(t.to(model.device, dtype=model.dtype).input_features)
        mel += processor.audio_length_per_tok
        start = mel * hop - win // 2

    t0 = time.perf_counter()
    with torch.no_grad():
        salida = model.generate(
            input_ids=entrada.input_ids,
            input_features=(t for t in trozos),
            num_delay_tokens=entrada.num_delay_tokens,
        )
    total = time.perf_counter() - t0

    return processor.batch_decode(salida, skip_special_tokens=True)[0], total

In [5]:
filas = []

for idioma in IDIOMAS:
    base, _, _ = IDIOMAS[idioma]

    for audio in tqdm(seleccion[idioma], desc=f"Voxtral {idioma}"):

        with open(base + "/%s/transcription.tok" % audio, "r", encoding="utf-8") as f:
            ref = f.read()

        audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
        duracion = len(audio_array) / sr

        for modo, fn in [("offline", transcribir_offline), ("streaming", transcribir_streaming)]:
            try:
                hyp, total = fn(audio_array)
                fallo = ""
            except Exception as e:
                hyp, total, fallo = "", np.nan, type(e).__name__
                print(f"  fallo {idioma}/{modo} en {audio}: {str(e)[:80]}")

            filas.append(dict(idioma=idioma, audio=audio, modo=modo, fallo=fallo,
                              duracion_audio_s=duracion, tiempo_total_s=total,
                              rtf=total / duracion, hypothesis=hyp, reference=ref))

data = pd.DataFrame(filas)
data.to_csv("voxtral_idiomas_raw.csv", index=False)
data.head()

Voxtral es:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_5878/848.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1162) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` 

Voxtral en:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_5878/848.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=887) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` t

Voxtral fr:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_5878/848.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=837) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` t

Voxtral de:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_5878/848.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(base + "/%s/audio_clip_diarization.m4a" % audio, sr=sr)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=537) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
/home/carru/venvs/voxtral/lib/python3.12/site-packages/transformers/generation/utils.py:1805: UserWarning: Using the model-agnostic default `max_length` (=1000000000) to control the generation length. We recommend setting `max_new_tokens` t

,idioma,audio,modo,fallo,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference
0,es,en.20090324.3.2-058,offline,,89.025312,85.258068,0.957683,"Yo quisiera, respecto al informe sobre microc...","Señora Presidenta , yo quisiera , respecto al ..."
1,es,en.20090324.3.2-058,streaming,,89.025312,79.806858,0.896451,"Yo quisiera, respecto al informe sobre microc...","Señora Presidenta , yo quisiera , respecto al ..."
2,es,en.20121025.28.4-369-000,offline,,65.015875,59.960027,0.922237,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar..."
3,es,en.20121025.28.4-369-000,streaming,,65.015875,60.437514,0.929581,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar..."
4,es,en.20101124.20.3-339,offline,,88.003687,88.025082,1.000243,Gracias también a la señora comisaria. Compar...,"Señor Presidente , señora Comisaria , comparto..."


In [6]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(t)) for t in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(t)) for t in data["reference"]]

data.to_csv("voxtral_idiomas_limpio.csv", index=False)
data.head()

,idioma,audio,modo,fallo,duracion_audio_s,tiempo_total_s,rtf,hypothesis,reference,hypothesis_clean,reference_clean
0,es,en.20090324.3.2-058,offline,,89.025312,85.258068,0.957683,"Yo quisiera, respecto al informe sobre microc...","Señora Presidenta , yo quisiera , respecto al ...",yo quisiera respecto al informe sobre microcr...,señora presidenta yo quisiera respecto al info...
1,es,en.20090324.3.2-058,streaming,,89.025312,79.806858,0.896451,"Yo quisiera, respecto al informe sobre microc...","Señora Presidenta , yo quisiera , respecto al ...",yo quisiera respecto al informe sobre microcr...,señora presidenta yo quisiera respecto al info...
2,es,en.20121025.28.4-369-000,offline,,65.015875,59.960027,0.922237,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar...",yo también quiero felicitar al ponente y al r...,señor presidente yo también quiero felicitar a...
3,es,en.20121025.28.4-369-000,streaming,,65.015875,60.437514,0.929581,Yo también quiero felicitar al ponente y al r...,"Señor Presidente , yo también quiero felicitar...",yo también quiero felicitar al ponente y al r...,señor presidente yo también quiero felicitar a...
4,es,en.20101124.20.3-339,offline,,88.003687,88.025082,1.000243,Gracias también a la señora comisaria. Compar...,"Señor Presidente , señora Comisaria , comparto...",gracias también a la señora comisaria compart...,señor presidente señora comisaria comparto el ...


In [7]:
# WER offline frente a streaming, por idioma
resumen = []

for (idioma, modo), g in data.groupby(["idioma", "modo"]):
    ok = g[g["fallo"] == ""]
    resumen.append(dict(
        idioma=idioma,
        modo=modo,
        audios=len(ok),
        fallos=int((g["fallo"] != "").sum()),
        wer=jiwer.wer(list(ok["reference_clean"]), list(ok["hypothesis_clean"])) * 100,
        rtf=ok["rtf"].mean(),
    ))

resumen = pd.DataFrame(resumen).pivot(index="idioma", columns="modo",
                                      values=["wer", "rtf", "fallos"])
resumen.to_csv("voxtral_idiomas_resumen.csv")
resumen

wer                  rtf            fallos          
modo      offline  streaming   offline streaming offline streaming
idioma                                                            
de      13.335717  13.371469  0.942010  0.939384     0.0       0.0
en      11.696917  11.696917  0.962085  0.981261     0.0       0.0
es       7.673797   7.673797  0.945079  0.943349     0.0       0.0
fr       7.678480   7.652799  0.924667  0.937628     0.0       0.0

In [8]:
# Degradación al pasar de offline a streaming
for idioma in IDIOMAS:
    g = data[data.idioma == idioma]
    off = g[(g.modo == "offline") & (g.fallo == "")]
    stm = g[(g.modo == "streaming") & (g.fallo == "")]
    if not len(off) or not len(stm):
        continue
    w_off = jiwer.wer(list(off.reference_clean), list(off.hypothesis_clean)) * 100
    w_stm = jiwer.wer(list(stm.reference_clean), list(stm.hypothesis_clean)) * 100
    print(f"{idioma}: offline {w_off:5.2f} %  ->  streaming {w_stm:5.2f} %  "
          f"(degradación {w_stm - w_off:+.2f} puntos, x{w_stm / w_off:.2f})")

es: offline  7.67 %  ->  streaming  7.67 %  (degradación +0.00 puntos, x1.00)
en: offline 11.70 %  ->  streaming 11.70 %  (degradación +0.00 puntos, x1.00)
fr: offline  7.68 %  ->  streaming  7.65 %  (degradación -0.03 puntos, x1.00)
de: offline 13.34 %  ->  streaming 13.37 %  (degradación +0.04 puntos, x1.00)
